# Genomic regulation of chemo-mechanical stability in plant-derived extracellular vesicles: a multiscale model of composite reinforcement
### Jagir R. Hussan, Anand Rampadarath, David P. Nickerson, Peter J. Hunter
### August, 2026

This notebook recomputes **every quantitative value cited in the manuscript** directly from the simulation CSV outputs.

**Inputs** (generated by the SCG-MD simulation scripts):
- `kev_mechanics.csv` — from `kev_mechanics.py` (pH sweep: 2.5, 6.8, 7.4 across State A / B / C)
- `kev_seeded_vs_spontaneous.csv` — from `kev_seeded_vs_spontaneous.py` (Seeded vs Spontaneous topology, State A / B, pH 6.8)

**Manuscript sections covered:**
1. Abstract — rupture tension, % increase, topology gap, surface-area fraction
2. §3.1 Mechanical hierarchy — $\gamma_{crit}$ (SEM), % increase, Cohen's $d$
3. §3.1 Role of lipid organisation — Seeded vs Spontaneous, % gap
4. §3.2 Topological reinforcement — $S_{max}$, monolithic vs composite
5. §3.3 Acid shock — Ripening stiffening %, Defence homeostasis %
6. §2.2 Force fields — $K_{eff}$ values, $k_{bond}$ relation
7. Statistical Tables 1–4 — rupture tension, percolation, Moran's $I$, Welch $t$-tests

> **Notation:** the CSV column names are kept stable for backwards-compatibility; this notebook maps them to the manuscript symbols:
> - `Rupture_Tension_mNm` → $\gamma_{crit}$ (critical rupture tension, mN m$^{-1}$)
> - `Percolation` → $S_{max}$ (rigid-domain percolation fraction)
> - `Heterogeneity` → Moran's $I$ (spatial-autocorrelation index)
> - `Ka_Input_mNm` → $K_{eff}$ (network area-compressibility modulus, mN m$^{-1}$)
> - `Mission` keys: `STATE_A_RIPENING` → State A (Ripening), `STATE_B_DEFENSE` → State B (Defence)
> - `Mode` keys: `SEEDED` → Seeded (Active sorting), `SPONTANEOUS` → Spontaneous (Entropic mixing)


---
## 1. Setup and data loading

In [15]:
from IPython.display import display, Markdown, HTML
import pandas as pd
import numpy as np
from scipy import stats

# -- Load data ---------------------------------------------------------------
dm = pd.read_csv('kev_mechanics.csv')
ds = pd.read_csv('kev_seeded_vs_spontaneous.csv')

# Drop any error rows (failed replicates have NaN in the value columns)
dm = dm[dm['Rupture_Tension_mNm'].notna()].copy()
ds = ds[ds['Rupture_Tension_mNm'].notna()].copy()

display(Markdown(rf"""
**`kev_mechanics.csv`** — {len(dm)} rows total

| State | Rows |
|:------|-----:|
| State A (Ripening) | {len(dm[dm.Mission == 'STATE_A_RIPENING'])} |
| State B (Defence) | {len(dm[dm.Mission == 'STATE_B_DEFENSE'])} |
| State C (exploratory) | {len(dm[dm.Mission == 'STATE_C_STRESS'])} |

**`kev_seeded_vs_spontaneous.csv`** — {len(ds)} rows total

| Condition | Rows |
|:----------|-----:|
| Seeded A | {len(ds[(ds.Mission == 'STATE_A_RIPENING') & (ds.Mode == 'SEEDED')])} |
| Seeded B | {len(ds[(ds.Mission == 'STATE_B_DEFENSE') & (ds.Mode == 'SEEDED')])} |
| Spontaneous A | {len(ds[(ds.Mission == 'STATE_A_RIPENING') & (ds.Mode == 'SPONTANEOUS')])} |
| Spontaneous B | {len(ds[(ds.Mission == 'STATE_B_DEFENSE') & (ds.Mode == 'SPONTANEOUS')])} |
"""))


**`kev_mechanics.csv`** — 450 rows total

| State | Rows |
|:------|-----:|
| State A (Ripening) | 150 |
| State B (Defence) | 150 |
| State C (exploratory) | 150 |

**`kev_seeded_vs_spontaneous.csv`** — 200 rows total

| Condition | Rows |
|:----------|-----:|
| Seeded A | 50 |
| Seeded B | 50 |
| Spontaneous A | 50 |
| Spontaneous B | 50 |


### Helper functions

In [16]:
def mean_sem(series):
    """Return (mean, SEM) of a pandas Series."""
    n = len(series)
    return series.mean(), series.std(ddof=1) / np.sqrt(n)

def mean_sd_ci(series):
    """Return (mean, SD, 95% CI) of a pandas Series."""
    n = len(series)
    m = series.mean()
    sd = series.std(ddof=1)
    se = sd / np.sqrt(n)
    ci = stats.t.interval(0.95, n - 1, loc=m, scale=se)
    return m, sd, ci

def cohens_d(group1, group2):
    """Cohen's d (pooled-SD standardisation): effect size of group1 vs group2."""
    n1, n2 = len(group1), len(group2)
    v1, v2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_sd

def welch_t(g1, g2):
    """Welch's two-sample t-test: returns (delta_mean, t_stat, p_value, cohens_d)."""
    delta = np.mean(g1) - np.mean(g2)
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    d = cohens_d(g1, g2)
    return delta, t, p, d

---
## 2. Abstract values

The abstract cites four headline numbers: rigid-domain surface fraction, critical rupture tension, % increase over Ripening, and the topology gap (Seeded vs Spontaneous).

In [17]:
# -- Abstract: Seeded State A (Ripening) and State B (Defence) at pH 6.8 -------
sA = ds[(ds.Mission == 'STATE_A_RIPENING') & (ds.Mode == 'SEEDED')]['Rupture_Tension_mNm']
sB = ds[(ds.Mission == 'STATE_B_DEFENSE')  & (ds.Mode == 'SEEDED')]['Rupture_Tension_mNm']
spB = ds[(ds.Mission == 'STATE_B_DEFENSE') & (ds.Mode == 'SPONTANEOUS')]['Rupture_Tension_mNm']
sB_perc = ds[(ds.Mission == 'STATE_B_DEFENSE') & (ds.Mode == 'SEEDED')]['Percolation']

gamma_B_mean, gamma_B_sem = mean_sem(sB)
pct_increase = (sB.mean() - sA.mean()) / sA.mean() * 100
topology_gap = (sB.mean() - spB.mean()) / spB.mean() * 100

display(Markdown(rf"""
### Abstract — headline values

| Quantity | Value | Source |
|:---------|:------|:-------|
| Rigid domain surface fraction ($S_{{max}}$, Seeded Defence) | **{sB_perc.mean()*100:.1f}%** | `kev_seeded_vs_spontaneous.csv` |
| Critical rupture tension $\gamma_{{crit}}$ (Defence, Seeded) | **{gamma_B_mean:.1f} ± {gamma_B_sem:.1f} mN m$^{{-1}}$** (SEM) | `kev_seeded_vs_spontaneous.csv` |
| Increase over Ripening (Seeded B vs Seeded A) | **{pct_increase:.0f}%** | Seeded B vs Seeded A |
| Topology gap (Seeded B vs Spontaneous B) | **{topology_gap:.0f}%** | Seeded B vs Spont B |
"""))


### Abstract — headline values

| Quantity | Value | Source |
|:---------|:------|:-------|
| Rigid domain surface fraction ($S_{max}$, Seeded Defence) | **37.7%** | `kev_seeded_vs_spontaneous.csv` |
| Critical rupture tension $\gamma_{crit}$ (Defence, Seeded) | **366.6 ± 0.8 mN m$^{-1}$** (SEM) | `kev_seeded_vs_spontaneous.csv` |
| Increase over Ripening (Seeded B vs Seeded A) | **38%** | Seeded B vs Seeded A |
| Topology gap (Seeded B vs Spontaneous B) | **22%** | Seeded B vs Spont B |


---
## 3. Mechanical hierarchy of states

The manuscript reports $\gamma_{crit}$ for Ripening (A) and Defence (B) as mean ± SEM, the % increase, and Cohen's $d$.

In [18]:
# -- §3.1: gamma_crit for State A and State B (Seeded, pH 6.8) ---------------
gamma_A_mean, gamma_A_sem = mean_sem(sA)
gamma_B_mean, gamma_B_sem = mean_sem(sB)
d_seeded = cohens_d(sB, sA)
delta_s, t_s, p_s, _ = welch_t(sB, sA)

display(Markdown(rf"""
### §3.1 Mechanical hierarchy

| Quantity | Value |
|:---------|:------|
| Ripening (State A, Seeded) $\gamma_{{crit}}$ | {gamma_A_mean:.1f} ± {gamma_A_sem:.2f} mN m$^{{-1}}$ (SEM) |
| Defence (State B, Seeded) $\gamma_{{crit}}$ | {gamma_B_mean:.1f} ± {gamma_B_sem:.2f} mN m$^{{-1}}$ (SEM) |
| Increase | **{pct_increase:.0f}%** |
| Cohen's $d$ (Defence vs Ripening) | **{d_seeded:.1f}** |
| Welch $t$-test | $t$ = {t_s:.2f}, $p$ = {p_s:.2e} |
"""))


### §3.1 Mechanical hierarchy

| Quantity | Value |
|:---------|:------|
| Ripening (State A, Seeded) $\gamma_{crit}$ | 265.9 ± 0.08 mN m$^{-1}$ (SEM) |
| Defence (State B, Seeded) $\gamma_{crit}$ | 366.6 ± 0.76 mN m$^{-1}$ (SEM) |
| Increase | **38%** |
| Cohen's $d$ (Defence vs Ripening) | **26.4** |
| Welch $t$-test | $t$ = 131.86, $p$ = 3.39e-65 |


---
## 4. Role of lipid organisation

Compares Seeded vs Spontaneous for State B (Defence). The manuscript cites the two mean tensions and the % gap.

In [19]:
# -- §3.1 organisation paragraph: Seeded B vs Spontaneous B ------------------
delta_org, t_org, p_org, d_org = welch_t(sB, spB)

display(Markdown(rf"""
### §3.1 Role of lipid organisation

| Quantity | Value |
|:---------|:------|
| Seeded Defence $\gamma_{{crit}}$ | {sB.mean():.1f} mN m$^{{-1}}$ |
| Spontaneous Defence $\gamma_{{crit}}$ | {spB.mean():.1f} mN m$^{{-1}}$ |
| Performance gap | **{topology_gap:.0f}%** |
| Welch $t$-test | $t$ = {t_org:.2f}, $p$ = {p_org:.2e} |
"""))


### §3.1 Role of lipid organisation

| Quantity | Value |
|:---------|:------|
| Seeded Defence $\gamma_{crit}$ | 366.6 mN m$^{-1}$ |
| Spontaneous Defence $\gamma_{crit}$ | 301.5 mN m$^{-1}$ |
| Performance gap | **22%** |
| Welch $t$-test | $t$ = 56.62, $p$ = 8.11e-76 |


---
## 5. Topological reinforcement mechanism

Reports $S_{max}$ (mean ± SD) for Seeded Defence, the Spontaneous monolithic value, and the % by which the monolithic structure is weaker.

In [20]:
# -- §3.2: S_max for Seeded Defence and Spontaneous Defence ------------------
sB_smax_mean = sB_perc.mean() * 100
sB_smax_sd = sB_perc.std(ddof=1) * 100
spB_perc = ds[(ds.Mission == 'STATE_B_DEFENSE') & (ds.Mode == 'SPONTANEOUS')]['Percolation']
spB_smax_mean = spB_perc.mean() * 100
pct_weaker = (sB.mean() - spB.mean()) / sB.mean() * 100

display(Markdown(rf"""
### §3.2 Topological reinforcement

| Quantity | Value |
|:---------|:------|
| Seeded Defence $S_{{max}}$ | {sB_smax_mean:.1f}% ± {sB_smax_sd:.1f}% (mean ± SD) |
| Spontaneous Defence $S_{{max}}$ (monolithic) | {spB_smax_mean:.1f}% |
| Monolithic weaker than composite by | **{pct_weaker:.0f}%** |
"""))


### §3.2 Topological reinforcement

| Quantity | Value |
|:---------|:------|
| Seeded Defence $S_{max}$ | 37.7% ± 9.3% (mean ± SD) |
| Spontaneous Defence $S_{max}$ (monolithic) | 49.6% |
| Monolithic weaker than composite by | **18%** |


---
## 6. Acid shock and mechanical homeostasis

Compares pH 6.8 vs pH 2.5 for both states. Ripening stress-stiffens (jamming); Defence exhibits homeostasis. Source: `kev_mechanics.csv`.

In [21]:
# -- §3.3: pH 6.8 vs pH 2.5 from kev_mechanics.csv ----------------------------
mA_68 = dm[(dm.Mission == 'STATE_A_RIPENING') & (dm.pH == 6.8)]['Rupture_Tension_mNm']
mA_25 = dm[(dm.Mission == 'STATE_A_RIPENING') & (dm.pH == 2.5)]['Rupture_Tension_mNm']
mB_68 = dm[(dm.Mission == 'STATE_B_DEFENSE')  & (dm.pH == 6.8)]['Rupture_Tension_mNm']
mB_25 = dm[(dm.Mission == 'STATE_B_DEFENSE')  & (dm.pH == 2.5)]['Rupture_Tension_mNm']

rip_delta = (mA_25.mean() - mA_68.mean()) / mA_68.mean() * 100
def_delta = (mB_25.mean() - mB_68.mean()) / mB_68.mean() * 100

claim_true = def_delta < 2.5
claim_str = " True" if claim_true else " False"

display(Markdown(rf"""
### §3.3 Acid shock (pH 6.8 → 1.3)

| State | pH 6.8 (mN m$^{{-1}}$) | pH 2.5 (mN m$^{{-1}}$) | $\Delta$ | Interpretation |
|:------|------:|------:|------:|:----------------|
| Ripening (A) | {mA_68.mean():.1f} | {mA_25.mean():.1f} | **+{rip_delta:.1f}%** | Stress-stiffening (jamming) |
| Defence (B) | {mB_68.mean():.1f} | {mB_25.mean():.1f} | **+{def_delta:.1f}%** | Mechanical homeostasis |

> **Fig 5 caption check:** claim r"$\Delta = 1.3\%$" — actual Defence $\Delta$ = {def_delta:.1f}%. The claim is **{claim_str}** .
"""))


### §3.3 Acid shock (pH 6.8 → 1.3)

| State | pH 6.8 (mN m$^{-1}$) | pH 2.5 (mN m$^{-1}$) | $\Delta$ | Interpretation |
|:------|------:|------:|------:|:----------------|
| Ripening (A) | 278.8 | 297.1 | **+6.6%** | Stress-stiffening (jamming) |
| Defence (B) | 455.5 | 461.4 | **+1.3%** | Mechanical homeostasis |

> **Fig 5 caption check:** claim r"$\Delta = 1.3\%$" — actual Defence $\Delta$ = 1.3%. The claim is ** True** .


---
## 7. Force fields — $K_{eff}$ values

The manuscript cites the network modulus $K_{eff}$ for the high-barrier and fluid homeostatic states. These are computed from Eq. 1: $K_{eff} = K_{base} + \alpha G_{SMT} - \beta G_{FAD}$.

In [22]:
# -- §2.2: K_eff from Eq. 1 and verified against the CSV ---------------------
K_base, alpha, beta = 240.0, 120.0, 60.0
profiles = {
    'STATE_A_RIPENING': {'SMT': 0.5, 'FAD': 3.0, 'label': 'State A (Ripening)'},
    'STATE_B_DEFENSE':  {'SMT': 4.0, 'FAD': 0.5, 'label': 'State B (Defence)'},
    'STATE_C_STRESS':   {'SMT': 2.0, 'FAD': 1.5, 'label': 'State C (exploratory)'},
}

rows_md = ""
for mission, g in profiles.items():
    keff = K_base + alpha * g['SMT'] - beta * g['FAD']
    keff_csv = dm[dm.Mission == mission]['Ka_Input_mNm'].mean()
    match = "Match" if abs(keff - keff_csv) < 0.1 else "Drift"
    rows_md += f"| {g['label']} | 240 + 120×{g['SMT']:.1f} − 60×{g['FAD']:.1f} | **{keff:.0f} mN m⁻¹** | {keff_csv:.0f} | {match} |\n"

display(Markdown(rf"""
### §2.2 $K_{{eff}}$ (network modulus)

**Eq. 1:** $K_{{eff}} = K_{{base}} + \alpha \, G_{{SMT}} - \beta \, G_{{FAD}}$

$K_{{base}} = {K_base:.0f}$, $\alpha = {alpha:.0f}$, $\beta = {beta:.0f}$ (mN m$^{{-1}}$)

| State | Calculation | $K_{{eff}}$ | CSV value | Match |
|:------|:------------|------:|------:|:-----:|
{rows_md.rstrip()}

$k_{{bond}} \approx 0.5 \, K_{{eff}}$ (Seung & Nelson, 1988)
"""))


### §2.2 $K_{eff}$ (network modulus)

**Eq. 1:** $K_{eff} = K_{base} + \alpha \, G_{SMT} - \beta \, G_{FAD}$

$K_{base} = 240$, $\alpha = 120$, $\beta = 60$ (mN m$^{-1}$)

| State | Calculation | $K_{eff}$ | CSV value | Match |
|:------|:------------|------:|------:|:-----:|
| State A (Ripening) | 240 + 120×0.5 − 60×3.0 | **120 mN m⁻¹** | 120 | Match |
| State B (Defence) | 240 + 120×4.0 − 60×0.5 | **690 mN m⁻¹** | 690 | Match |
| State C (exploratory) | 240 + 120×2.0 − 60×1.5 | **390 mN m⁻¹** | 390 | Match |

$k_{bond} \approx 0.5 \, K_{eff}$ (Seung & Nelson, 1988)


---
## 8. Statistical Table 1 — Rupture tension ($\gamma_{crit}$)

Appendix table `1`: $n=50$ per group; mean ± SD, 95% CI. Source: `kev_seeded_vs_spontaneous.csv`.

In [23]:
# -- Table 1: Rupture tension statistics -------------------------------------
modes = ['SEEDED', 'SPONTANEOUS']
missions_t = [('STATE_A_RIPENING', 'A (Rip.)'), ('STATE_B_DEFENSE', 'B (Def.)')]

rows_html = ""
for mode in modes:
    for mission, label in missions_t:
        data = ds[(ds.Mode == mode) & (ds.Mission == mission)]['Rupture_Tension_mNm']
        m, sd, ci = mean_sd_ci(data)
        rows_html += f"<tr><td><tt>{mode}</tt></td><td>{label}</td><td>{m:.1f}</td><td>{sd:.1f}</td><td>[{ci[0]:.1f}, {ci[1]:.1f}]</td></tr>\n"
    if mode == 'SEEDED':
        rows_html += '<tr style="border-top: 2px solid #ddd;"></tr>\n'

display(HTML(rf"""
<h4>Table 1: Rupture tension statistics ($\gamma_{{crit}}$)</h4>
<p><em>n = 50 per group; mean ± SD, 95% CI</em></p>
<table>
<thead><tr><th>Init.</th><th>State</th><th>$\gamma$ (mN m⁻¹)</th><th>SD</th><th>95% CI</th></tr></thead>
<tbody>
{rows_html.rstrip()}
</tbody>
</table>
"""))

Init.,State,$\gamma$ (mN m⁻¹),SD,95% CI
SEEDED,A (Rip.),265.9,0.5,"[265.7, 266.0]"
SEEDED,B (Def.),366.6,5.4,"[365.1, 368.1]"
SPONTANEOUS,A (Rip.),236.2,0.8,"[236.0, 236.4]"
SPONTANEOUS,B (Def.),301.5,6.1,"[299.7, 303.2]"


---
## 9. Statistical Table 2 — Percolation fraction ($S_{max}$)

Appendix table `2`: $n=50$ per group; mean ± SD, 95% CI.

In [24]:
# -- Table 2: Percolation fraction (S_max) ------------------------------------
rows_html = ""
for mode in modes:
    for mission, label in missions_t:
        data = ds[(ds.Mode == mode) & (ds.Mission == mission)]['Percolation']
        m, sd, ci = mean_sd_ci(data)
        rows_html += f"<tr><td><tt>{mode}</tt></td><td>{label}</td><td>{m:.3f}</td><td>{sd:.3f}</td><td>[{ci[0]:.3f}, {ci[1]:.3f}]</td></tr>\n"
    if mode == 'SEEDED':
        rows_html += '<tr style="border-top: 2px solid #ddd;"></tr>\n'

display(HTML(rf"""
<h4>Table 2: Percolation fraction statistics ($S_{{max}}$)</h4>
<p><em>n = 50 per group; mean ± SD, 95% CI</em></p>
<table>
<thead><tr><th>Init.</th><th>State</th><th>Mean</th><th>SD</th><th>95% CI</th></tr></thead>
<tbody>
{rows_html.rstrip()}
</tbody>
</table>
"""))

Init.,State,Mean,SD,95% CI
SEEDED,A (Rip.),0.019,0.006,"[0.017, 0.021]"
SEEDED,B (Def.),0.377,0.093,"[0.350, 0.403]"
SPONTANEOUS,A (Rip.),0.005,0.001,"[0.005, 0.006]"
SPONTANEOUS,B (Def.),0.496,0.002,"[0.495, 0.496]"


---
## 10. Statistical Table 3 — Moran's $I$

Appendix table `3`: $n=50$ per group; mean ± SD, 95% CI.

In [25]:
# -- Table 3: Moran's I (Heterogeneity) --------------------------------------
rows_html = ""
for mode in modes:
    for mission, label in missions_t:
        data = ds[(ds.Mode == mode) & (ds.Mission == mission)]['Heterogeneity']
        m, sd, ci = mean_sd_ci(data)
        rows_html += f"<tr><td><tt>{mode}</tt></td><td>{label}</td><td>{m:.3f}</td><td>{sd:.3f}</td><td>[{ci[0]:.3f}, {ci[1]:.3f}]</td></tr>\n"
    if mode == 'SEEDED':
        rows_html += '<tr style="border-top: 2px solid #ddd;"></tr>\n'

display(HTML(rf"""
<h4>Table 3: Moran's $I$ statistics</h4>
<p><em>n = 50 per group; mean ± SD, 95% CI</em></p>
<table>
<thead><tr><th>Init.</th><th>State</th><th>$I$</th><th>SD</th><th>95% CI</th></tr></thead>
<tbody>
{rows_html.rstrip()}
</tbody>
</table>
"""))

Init.,State,$I$,SD,95% CI
SEEDED,A (Rip.),0.620,0.030,"[0.612, 0.629]"
SEEDED,B (Def.),0.626,0.020,"[0.620, 0.631]"
SPONTANEOUS,A (Rip.),0.004,0.010,"[0.001, 0.007]"
SPONTANEOUS,B (Def.),0.003,0.008,"[0.001, 0.006]"


---
## 11. Statistical Table 4 — Welch two-sample $t$-tests

Appendix table `4`: Defence vs Ripening by initialisation mode.

In [26]:
# -- Table 4: Welch t-tests (Defence vs Ripening) ----------------------------
rows_html = ""
for mode in modes:
    g_def = ds[(ds.Mode == mode) & (ds.Mission == 'STATE_B_DEFENSE')]['Rupture_Tension_mNm']
    g_rip = ds[(ds.Mode == mode) & (ds.Mission == 'STATE_A_RIPENING')]['Rupture_Tension_mNm']
    delta, t, p, d = welch_t(g_def, g_rip)
    rows_html += f"<tr><td><tt>{mode}</tt></td><td>{delta:.1f}</td><td>{t:.2f}</td><td>{p:.2e}</td><td>{d:.2f}</td></tr>\n"

display(HTML(rf"""
<h4>Table 4: Welch two-sample $t$-tests (Defence vs Ripening)</h4>
<table>
<thead><tr><th>Mode</th><th>$\Delta$ (mN m⁻¹)</th><th>$t$</th><th>$p$</th><th>Cohen's $d$</th></tr></thead>
<tbody>
{rows_html.rstrip()}
</tbody>
</table>
"""))

Mode,$\Delta$ (mN m⁻¹),$t$,$p$,Cohen's $d$
SEEDED,100.7,131.86,3.39e-65,26.37
SPONTANEOUS,65.3,75.01,1.87e-53,15.00


---
## 12. Summary — all manuscript values at a glance

A consolidated table of every number cited in the manuscript, its source, and the data-computed value.

In [27]:
# -- Consolidated summary ----------------------------------------------------
rows_html = ""
rows = [
    ("Abstract", "Rigid surface fraction ($S_{max}$)", f"{sB_perc.mean()*100:.0f}%", "kev_seeded_vs_spontaneous.csv"),
    ("Abstract", "$\\gamma_{crit}$ (Defence, Seeded)", f"{gamma_B_mean:.1f} ± {gamma_B_sem:.1f} mN m⁻¹ (SEM)", "kev_seeded_vs_spontaneous.csv"),
    ("Abstract", "Increase over Ripening", f"{pct_increase:.0f}%", "Seeded B vs Seeded A"),
    ("Abstract", "Topology gap (Seeded vs Spont)", f"{topology_gap:.0f}%", "Seeded B vs Spont B"),
    ("§3.1", "Ripening $\\gamma_{crit}$ (SEM)", f"{gamma_A_mean:.1f} ± {gamma_A_sem:.2f} mN m⁻¹", "Seeded A"),
    ("§3.1", "Defence $\\gamma_{crit}$ (SEM)", f"{gamma_B_mean:.1f} ± {gamma_B_sem:.2f} mN m⁻¹", "Seeded B"),
    ("§3.1", "Cohen's $d$ (Defence vs Ripening)", f"{d_seeded:.1f}", "Seeded B vs A"),
    ("§3.1 org", "Seeded Defence $\\gamma_{crit}$", f"{sB.mean():.1f} mN m⁻¹", "Seeded B"),
    ("§3.1 org", "Spontaneous Defence $\\gamma_{crit}$", f"{spB.mean():.1f} mN m⁻¹", "Spont B"),
    ("§3.1 org", "Performance gap", f"{topology_gap:.0f}%", "Seeded B vs Spont B"),
    ("§3.2", "Seeded $S_{max}$ (mean ± SD)", f"{sB_smax_mean:.1f}% ± {sB_smax_sd:.1f}%", "Seeded B"),
    ("§3.2", "Spontaneous $S_{max}$ (monolithic)", f"{spB_smax_mean:.1f}%", "Spont B"),
    ("§3.2", "Monolithic weaker by", f"{pct_weaker:.0f}%", "Spont B vs Seeded B"),
    ("§3.3", "Ripening acid shock $\\Delta$", f"+{rip_delta:.1f}%", "mechanics pH 6.8 → 2.5"),
    ("§3.3", "Defence acid shock $\\Delta$", f"+{def_delta:.1f}%", "mechanics pH 6.8 → 2.5"),
    ("§2.2", r"$K_{eff}$ State B (high-barrier)", f"{240 + 120*4.0 - 60*0.5:.0f} mN m⁻¹", "Eq. 1"),
    ("§2.2", r"$K_{eff}$ State A (homeostatic)", f"{240 + 120*0.5 - 60*3.0:.0f} mN m⁻¹", "Eq. 1"),
]
for sec, qty, val, src in rows:
    rows_html += f"<tr><td>{sec}</td><td>{qty}</td><td><b>{val}</b></td><td><code>{src}</code></td></tr>\n"

display(HTML(rf"""
<h4>Consolidated summary — all manuscript values</h4>
<table>
<thead><tr><th>Section</th><th>Quantity</th><th>Value</th><th>Source</th></tr></thead>
<tbody>
{rows_html.rstrip()}
</tbody>
</table>
<p><em>Tables 1–4 are rendered in full in sections 8–11 above.</em></p>
"""))

Section,Quantity,Value,Source
Abstract,Rigid surface fraction ($S_{max}$),38%,kev_seeded_vs_spontaneous.csv
Abstract,"$\gamma_{crit}$ (Defence, Seeded)",366.6 ± 0.8 mN m⁻¹ (SEM),kev_seeded_vs_spontaneous.csv
Abstract,Increase over Ripening,38%,Seeded B vs Seeded A
Abstract,Topology gap (Seeded vs Spont),22%,Seeded B vs Spont B
§3.1,Ripening $\gamma_{crit}$ (SEM),265.9 ± 0.08 mN m⁻¹,Seeded A
§3.1,Defence $\gamma_{crit}$ (SEM),366.6 ± 0.76 mN m⁻¹,Seeded B
§3.1,Cohen's $d$ (Defence vs Ripening),26.4,Seeded B vs A
§3.1 org,Seeded Defence $\gamma_{crit}$,366.6 mN m⁻¹,Seeded B
§3.1 org,Spontaneous Defence $\gamma_{crit}$,301.5 mN m⁻¹,Spont B
§3.1 org,Performance gap,22%,Seeded B vs Spont B


---
## Notes

- **Source data:** the pH-sweep values (§3.3) come from `kev_mechanics.csv`, which uses `kev_mechanics.py`'s Seeded-only topology across pH 2.5 / 6.8 / 7.4. The topology comparison values (abstract, §3.1, §3.2, Tables 1–4) come from `kev_seeded_vs_spontaneous.csv`, which uses a fixed pH 6.8.
- **State C** (`STATE_C_STRESS`) is an exploratory intermediate archetype retained in `kev_mechanics.py` for parameter-space coverage; it is **not** reported in the manuscript.
- **Re-run note:** `kev_seeded_vs_spontaneous.py` uses Debye length $\lambda_D = 1.0$ nm. If the CSV was generated with a different value, all topology-comparison numbers will change.
